# 03 — Derivacion independiente de los indicadores de tiempo, costo y alcance

**Objetivo especifico que responde:** nucleo de OE-03 y operacionalizacion directa del marco teorico de la **restriccion de hierro** (`InformesRev/AnteproyectoAndres.md`, numeral 2.2.2 y 3.3).

**Regla de blindaje de autoria (recordatorio):** las formulas de este notebook se derivan **exclusivamente** de la definicion textual del anteproyecto. No se leyo, copio ni parafraseo la logica de `notebooks/07_depuracion_variables_modelo.ipynb` (que construye los targets binarios `tiempo`/`presupuesto`/`alcance` para el modelado predictivo del companero — codigo excluido). Las columnas `tiempo`, `presupuesto`, `alcance`, `sobrecosto_ratio`, `duracion_planificada_dias`, `ratio_extension_duracion` que ya existen en el dataset integrado **no se usan ni se leen** en este notebook.

**Definiciones citadas del anteproyecto (numeral 3.3):**
- *Desviacion de tiempo*: "diferencia entre plazo pactado y plazo real de ejecucion".
- *Desviacion de costo*: "diferencia entre valor inicial y valor final del contrato, expresada como porcentaje de adicion presupuestal".
- *Indice de cumplimiento del alcance*: "indicador ... que señala si el objeto contractual fue entregado en su totalidad, parcialmente o no fue entregado" — operacionalizado en el numeral 2.2.2 mediante "terminaciones anticipadas, suspensiones prolongadas o reducciones del objeto contratado".

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

REPO = Path(".").resolve().parents[1] if Path(".").resolve().name == "notebooks" else Path(".").resolve()
DATA_DIR = REPO / "analitica-descriptiva-andres" / "data"

df = pd.read_parquet(DATA_DIR / "muestra_analitica.parquet")
print("Muestra analitica cargada:", df.shape)


Muestra analitica cargada: (19194, 29)


## 1. Desviacion de tiempo (dias)

Se requiere el plazo pactado (planeado) en dias. La unica fuente cruda disponible para el plazo pactado es el campo de texto libre `duraci_n_del_contrato` (ej. `"4 Mes(es)"`, `"135 Dia(s)"`, `"No definido"`), que se parsea aqui de forma independiente (no se usa `duracion_planificada_dias`, que es la version ya parseada por el companero).

**Regla de conversion adoptada (propia, documentada):** `Dia(s)` se toma literal; `Mes(es)` se convierte multiplicando por 30 (aproximacion estandar de mes comercial, ya que SECOP no reporta meses calendario exactos); `No definido` o valores sin numero se tratan como dato faltante.

In [2]:
def parsear_duracion_dias(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto).strip()
    match = re.match(r"(\d+)\s*(Dia|Mes)", texto, flags=re.IGNORECASE)
    if not match:
        return np.nan
    numero, unidad = int(match.group(1)), match.group(2).lower()
    return numero * 30 if unidad == 'mes' else numero

df['plazo_pactado_dias'] = df['duraci_n_del_contrato'].apply(parsear_duracion_dias)
df['plazo_real_dias'] = (df['fecha_de_fin_del_contrato'] - df['fecha_de_inicio_del_contrato']).dt.days

print("plazo_pactado_dias -- nulos:", df['plazo_pactado_dias'].isna().sum(), f"({df['plazo_pactado_dias'].isna().mean()*100:.1f}%)")
print(df['plazo_pactado_dias'].describe())
print()
print("plazo_real_dias -- nulos:", df['plazo_real_dias'].isna().sum())
print(df['plazo_real_dias'].describe())


plazo_pactado_dias -- nulos: 2868 (14.9%)
count    16326.000000
mean       182.208686
std        503.048775
min          1.000000
25%         60.000000
50%        115.000000
75%        229.750000
max      33420.000000
Name: plazo_pactado_dias, dtype: float64

plazo_real_dias -- nulos: 0
count    19194.000000
mean       231.427477
std        279.928939
min          0.000000
25%         61.000000
50%        136.000000
75%        303.000000
max       7305.000000
Name: plazo_real_dias, dtype: float64


In [3]:
# Desviacion de tiempo = plazo real - plazo pactado (dias). Positivo = retraso; negativo = adelanto.
df['desviacion_tiempo_dias'] = df['plazo_real_dias'] - df['plazo_pactado_dias']

n_validos_tiempo = df['desviacion_tiempo_dias'].notna().sum()
print(f"Tamaño de muestra efectivo para desviacion de tiempo: {n_validos_tiempo} de {len(df)} "
      f"({n_validos_tiempo/len(df)*100:.1f}%) — el resto queda sin calcular por plazo pactado no parseable "
      f"('No definido' o formato no reconocido) o fechas faltantes.")
print()
print(df['desviacion_tiempo_dias'].describe())
print()
pct_con_retraso = (df['desviacion_tiempo_dias'] > 0).mean() * 100
print(f"Porcentaje de contratos (con dato valido) con plazo real mayor al pactado (retraso): {pct_con_retraso:.1f}%")


Tamaño de muestra efectivo para desviacion de tiempo: 16326 de 19194 (85.1%) — el resto queda sin calcular por plazo pactado no parseable ('No definido' o formato no reconocido) o fechas faltantes.

count    16326.000000
mean        35.694414
std        463.565735
min     -30072.000000
25%         -1.000000
50%          2.000000
75%         60.000000
max       1721.000000
Name: desviacion_tiempo_dias, dtype: float64

Porcentaje de contratos (con dato valido) con plazo real mayor al pactado (retraso): 51.2%


## 2. Desviacion de costo (%)

Formula literal del anteproyecto: `(valor_final - valor_inicial) / valor_inicial`, expresada como porcentaje. La unica aproximacion cruda disponible al "valor final" es `valor_pagado`. Se advierte una limitacion de cobertura real de este campo detectada en esta misma celda (ver salida).

In [4]:
cobertura_pago = (df['valor_pagado'] > 0).mean() * 100
print(f"Contratos con valor_pagado > 0 en la muestra analitica: {cobertura_pago:.1f}% "
      f"({(df['valor_pagado']>0).sum()} de {len(df)}).")
print("Los contratos con valor_pagado == 0 no permiten calcular una desviacion de costo real "
      "(no se puede distinguir 'sin sobrecosto' de 'pago no reportado por la entidad en SECOP').")
print("Por tanto, el indicador de desviacion de costo se calcula UNICAMENTE sobre el subconjunto con valor_pagado > 0,")
print("declarando explicitamente el tamaño de muestra efectivo reducido, tal como exige el estandar de rigor de este ejercicio.")


Contratos con valor_pagado > 0 en la muestra analitica: 45.2% (8681 de 19194).
Los contratos con valor_pagado == 0 no permiten calcular una desviacion de costo real (no se puede distinguir 'sin sobrecosto' de 'pago no reportado por la entidad en SECOP').
Por tanto, el indicador de desviacion de costo se calcula UNICAMENTE sobre el subconjunto con valor_pagado > 0,
declarando explicitamente el tamaño de muestra efectivo reducido, tal como exige el estandar de rigor de este ejercicio.


In [5]:
df['desviacion_costo_pct'] = np.where(
    df['valor_pagado'] > 0,
    (df['valor_pagado'] - df['valor_del_contrato']) / df['valor_del_contrato'] * 100,
    np.nan,
)

n_validos_costo = df['desviacion_costo_pct'].notna().sum()
print(f"Tamaño de muestra efectivo para desviacion de costo: {n_validos_costo} de {len(df)} "
      f"({n_validos_costo/len(df)*100:.1f}%).")
print()
print(df['desviacion_costo_pct'].describe())
print()
pct_con_sobrecosto = (df['desviacion_costo_pct'] > 0).mean() * 100
print(f"De los contratos con dato valido, {pct_con_sobrecosto:.1f}% presentan valor pagado superior al valor inicial (sobrecosto).")


Tamaño de muestra efectivo para desviacion de costo: 8681 de 19194 (45.2%).

count    8681.000000
mean       -7.926474
std        18.628127
min       -99.861778
25%        -2.879529
50%         0.000000
75%         0.000000
max        66.293805
Name: desviacion_costo_pct, dtype: float64

De los contratos con dato valido, 0.1% presentan valor pagado superior al valor inicial (sobrecosto).


## 3. Indice de cumplimiento de alcance (categorico)

Operacionalizacion propia, basada en el numeral 2.2.2 del anteproyecto ("terminaciones anticipadas, suspensiones prolongadas o reducciones del objeto contratado"), usando las columnas crudas agregadas de adiciones ya integradas por el pipeline compartido (`tiene_conclusion`, `tiene_suspension`, `tiene_cesion` — agregados en `notebooks/05`, no derivados por el companero en `07`/`08`):

- **No entregado**: `tiene_conclusion == 1` (terminacion anticipada registrada).
- **Entregado parcialmente**: sin terminacion anticipada, pero con `tiene_suspension == 1` o `tiene_cesion == 1` (ejecucion interrumpida o cedida a un tercero).
- **Entregado en su totalidad**: sin terminacion anticipada, sin suspension y sin cesion.

**Limitacion declarada:** no existe en el dataset integrado un indicador directo de "reduccion del objeto contratado" (cambio de alcance sin terminacion ni suspension), por lo que esta categoria no puede distinguirse por separado con los datos disponibles; queda documentada como limitacion y oportunidad de enriquecimiento futuro (ej. mediante analisis de texto de `objeto_del_contrato` vs. actas de liquidacion, fuera del alcance de este ejercicio).

In [6]:
def clasificar_alcance(row):
    if row['tiene_conclusion'] == 1:
        return 'No entregado'
    if row['tiene_suspension'] == 1 or row['tiene_cesion'] == 1:
        return 'Entregado parcialmente'
    return 'Entregado en su totalidad'

df['indice_alcance'] = df.apply(clasificar_alcance, axis=1)
conteo_alcance = df['indice_alcance'].value_counts()
print(conteo_alcance)
print()
print((conteo_alcance / len(df) * 100).round(1).astype(str) + '%')


indice_alcance
Entregado en su totalidad    10961
No entregado                  4323
Entregado parcialmente        3910
Name: count, dtype: int64

indice_alcance
Entregado en su totalidad    57.1%
No entregado                 22.5%
Entregado parcialmente       20.4%
Name: count, dtype: object


In [7]:
# --- Persistencia de los indicadores propios ---
columnas_salida = [
    'id_contrato', 'proceso_de_compra', 'codigo_proveedor', 'codigo_entidad',
    'territorio_id', 'modalidad_id',
    'plazo_pactado_dias', 'plazo_real_dias', 'desviacion_tiempo_dias',
    'valor_del_contrato', 'valor_pagado', 'desviacion_costo_pct',
    'tiene_conclusion', 'tiene_suspension', 'tiene_cesion', 'indice_alcance',
    'n_adicion_valor', 'n_modificaciones_total', 'dias_adicionados',
    'tipo_contratista', 'es_pyme', 'sector', 'rama', 'entidad_centralizada',
    'departamento', 'ciudad', 'orden', 'modalidad_de_contratacion',
]
indicadores = df[columnas_salida].copy()
indicadores.to_parquet(DATA_DIR / "indicadores_restriccion_hierro.parquet", index=False)
print("Guardado: indicadores_restriccion_hierro.parquet —", indicadores.shape)


Guardado: indicadores_restriccion_hierro.parquet — (19194, 28)


## 4. Sintesis de hallazgos de este notebook

- Se derivaron de forma **completamente independiente** (sin consultar codigo del companero) los tres indicadores de la restriccion de hierro exigidos por el anteproyecto: desviacion de tiempo (dias, continua), desviacion de costo (%, continua) e indice de cumplimiento de alcance (categorico, 3 niveles).
- El tamaño de muestra efectivo difiere entre indicadores (tiempo, costo y alcance no comparten exactamente la misma cobertura de datos validos), lo cual se declara explicitamente y debe respetarse en los notebooks siguientes (no combinar los tres indicadores asumiendo el mismo N sin verificarlo de nuevo).
- Se identifico una limitacion de cobertura relevante en `valor_pagado` (mas de la mitad de los contratos de la muestra analitica, ya filtrada a estados finalizados, registran valor pagado en cero), que se interpreta como una brecha de reporte de SECOP, no como ausencia real de pago, y se declara como limitacion metodologica del indicador de costo.
- **Hallazgo relevante para discutir en la sintesis final (notebook 08):** la tasa de sobrecosto obtenida aqui (0.1% de los contratos con dato valido) es sustancialmente menor a la cifra citada en el planteamiento del problema del anteproyecto (~1 de cada 4 contratos, segun fuentes de control social citadas en el numeral 1.2). Una hipotesis plausible, sin haber consultado el codigo excluido del companero, es que `valor_del_contrato` en el dataset integrado refleje el valor **vigente al momento de la extraccion** (ya actualizado tras adiciones formales de valor), no el valor **originalmente pactado** en la firma; si ese es el caso, comparar `valor_pagado` contra `valor_del_contrato` subestima sistematicamente el sobrecosto real, porque el propio denominador ya absorbio buena parte de las adiciones. Esta hipotesis debe verificarse en trabajo futuro contrastando contra el valor inicial verdadero (no disponible como columna cruda independiente en este dataset integrado) y se declara aqui como limitacion metodologica explicita, no como conclusion definitiva.
- El indice de alcance categorico no pudo capturar la dimension de "reduccion del objeto contratado" por ausencia de una variable cruda especifica; se declara como limitacion y se recomienda como trabajo futuro.
- Se observan valores extremos en `desviacion_tiempo_dias` (minimo y maximo de varios miles de dias), heredados de inconsistencias ya documentadas en el dato fuente (duraciones o fechas absurdas de SECOP). Estos outliers **no se tratan en este notebook**; se abordan explicitamente con winsorizacion y estadisticos robustos en el notebook 04 (EDA univariado).